# 第3章 次の一語はサイコロで決まる：確率と温度

Σ・指数関数・softmaxの「なぜその式になるのか」から、文章生成ループを自作する

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：LLMの「次のトークンの点数（ロジット）」を取り出し、自分でsoftmax・温度・サンプリングを書いて1トークンずつ文章を作る自作 `generate`
- **身につく数学・理論**：ロジット（生の点数）と確率分布の2条件 ／ Σ（シグマ）記号の読み方とfor文との対応 ／ なぜ素朴な割り算や2乗ではダメなのか：指数関数 $e^x$ と softmax の理由 ／ 平行移動不変性とオーバーフロー対策 ／ サンプリング（累積区間くじ引き）と乱数シード ／ 温度（temperature）と greedy（貪欲法） ／ 自己回帰生成とテンソル shape の変化
- **新しく使うPython・PyTorch**：辞書のアンパック `**inputs` ／ `with torch.no_grad():` と `.to(model.device)` ／ 3次元テンソル `[1, T, V]` と `[0, -1]` ／ `dim=-1` と `.float()` と `.item()` ／ `import math`・`math.exp`・`random.seed` ／ `torch.multinomial` と `torch.cat`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（GPU：T4） |
| 所要時間目安 | 75〜100分 |
| 難易度 | ★★☆☆☆ |

> **⚠️ GPUランタイム設定**：メニューの「ランタイム」→「ランタイムのタイプを変更」→「**T4 GPU**」を選んでから上から順に実行してください。
> 無料枠の混雑・上限でGPUに接続できない場合は、時間をおくか [Kaggle Notebooks](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html) を使うか、各章のCPU短縮手順で進められます。接続切断に備え、学習済みモデルはこまめにGoogleドライブへ保存しましょう。

> **🎯 この章でつくるもの**
>
> この章が終わるころ、あなたは次の4つができるようになります。
>
> 1. `model(**inputs)`、`with torch.no_grad():`、`out.logits[0, -1]` の**入力・中間値・テンソルの形（shape）**を説明できる
> 2. $\sum$（シグマ）記号を普通の足し算と `for` 文に翻訳し、小さな数字で手計算できる
> 3. **なぜ**ロジットを確率に変えるときに素朴な割り算や2乗ではなく指数関数 $e^x$（softmax）を使うのかを説明し、オーバーフロー安全な `softmax` を自力実装できる
> 4. 温度つきサンプリングと自己回帰ループ（自作 `generate`）を動かして、温度による文章変化の理由を説明できる

> **🔗 これまでの復習とつながり：第1章・第2章とのつながり**
>
> - **第1章**：LLMは「次のトークンの確率」を予測し続け、毎回サイコロを振って言葉を選んでいることを見ました。
> - **第2章**：文章は `tok.encode(...)` でトークンIDの数字列になり、最後に `tok.decode(...)` で文章に戻ることを学びました。
>
> この章では、そのあいだに挟まっている **「モデルが出した生の点数（ロジット）→ softmaxで確率 → 温度調整 → サイコロで1トークン選ぶ → 末尾に足す」** を、数式の理由とコードの両面から完全に分解して自作します！

**ミナト**：第1章で同じ質問をしても返事が少しずつ変わった理由が気になってました。それと……この章から数式が出てきますよね？ Σとか $e^x$ とか、正直あまり覚えていなくて……。

**ボンゴ先生**：大歓迎だ！ 記号を覚えている前提には一切しないよ。「どんな問題を解きたいのか → 素朴な方法ではなぜダメなのか → 小さな数字で手計算 → 数式にまとめる → Pythonで確かめる」の順番で1つずつ登ろう。数式の意味が分かった瞬間、コードも一気にスッキリ見えるはずだ！

## 準備：モデルを直接読み込む

第1章では便利セットの `pipeline` を使いましたが、今回は中身を取り出すために**トークナイザー**と**モデル本体**を別々に読み込みます。ランタイムは「T4 GPU」にしておきましょう（CPUでも少し待てば動きます）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B-Instruct"
gpu = torch.cuda.is_available()
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name, dtype=torch.float16 if gpu else torch.float32, device_map="auto"
)
print("読み込み完了。置き場所：", model.device)

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
読み込み完了。置き場所： cuda:0
```

</details>

- `AutoModelForCausalLM` の `CausalLM` は「左から右へ次のトークンを予測する言語モデル（GPT系）」という意味です。
- `model.device` はモデルが置かれた機器を表す属性で、GPUの0番なら `cuda:0`、CPUなら `cpu` になります（**CUDA**はNVIDIA GPUで計算する仕組み）。

## モデルを直接読み込み、入力と出力の形を見る

まず「吾輩は猫である。名前は」をモデルに渡して、次のトークンに対する「生の点数表」を取り出します。本物のコードを実行する前に、初登場の書き方2つ（`**inputs` とテンソルの形）を**理解のための最小コード**で確認しましょう。

<details><summary><b>基礎の補足：理解のための最小コード：`model(**inputs)` の `**` と、3次元テンソル `[1, T, V]`</b>（クリックで開く）</summary>

**① 辞書のアンパック `**inputs` とは？**
関数を呼ぶとき `**辞書` と書くと、辞書の中身を「名前つき引数」に展開して渡してくれます。

```python no-run
def greet(name, times):
    return (name + "! ") * times

d = {"name": "猫", "times": 2}
print(greet(**d))   # greet(name="猫", times=2) とまったく同じ！ → 猫! 猫!
```

`tok(text, return_tensors="pt")` は `{"input_ids": テンソル, "attention_mask": テンソル}` という**辞書**を返します。だから `model(**inputs)` は `model(input_ids=..., attention_mask=...)` の短縮表記です（第7章で出てくるべき乗 `a ** 2` とは別の使い方です）。

**② `inputs` と `out.logits` の中身と形（shape）**
「吾輩は猫である。名前は」は6トークン（`T = 6`）だとします。

| 変数 | 型・形（`shape`） | 中身の意味 |
|---|---|---|
| `inputs["input_ids"]` | `Tensor`、形 `[1, 6]` | 1件の文章 × 6個のトークンID（`return_tensors="pt"` でPyTorchテンソルに変換） |
| `out.logits` | `Tensor`、形 `[1, 6, 151936]` | 1件の文章 × 各位置（0〜5） × 全151,936語ぶんの生の点数 |
| `out.logits[0, -1]` | `Tensor`、形 `[151936]` | **0番目の文章の、最後の位置（`-1`）**における全語彙の点数（＝「名前は」の**次**の予測！） |

</details>

では実際に動かして確かめましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
text = "吾輩は猫である。名前は"
inputs = tok(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model(**inputs)

logits = out.logits[0, -1]
print(logits.shape)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([151936])
```

</details>

- `.to(model.device)`：トークナイザーがCPU上に作った入力テンソルを、モデルと同じ機器（GPUなら `cuda:0`）へ移動します。場所が違うと計算エラーになります。
- `with torch.no_grad():`：学習時に必要な「逆伝播用の途中メモ」を保存しないモードです（第7章で理由を学びます）。推論ではメモ不要なので速く省メモリになります。
- `logits`（**ロジット**）：モデルが出力した各トークンの**生の点数（スコア）**です。プラスにもマイナスにもなり、上限も下限もありません（語彙数は151,665ですが、GPU計算の区切りの良さから枠が151,936取られています）。

点数の高い上位5個を見てみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
top = torch.topk(logits, 5)
for score, i in zip(top.values, top.indices):
    print(f"{tok.decode([int(i)])!r:8} 点数 {score.item():6.2f}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
'まだ'    点数  21.03
'ない'    点数  16.55
'、'      点数  15.22
'「'      点数  14.98
'無い'    点数  14.42
```

</details>

<details><summary><b>基礎の補足：`torch.topk`・`zip`・`.item()`・f文字列書式</b>（クリックで開く）</summary>

- `torch.topk(logits, 5)`：大きい順に5個の点数（`top.values`、形 `[5]`）とそのID（`top.indices`、形 `[5]`）をセットで返します。
- `zip(A, B)`：2つの並びを横に並べて1組ずつ取り出します（1位の点数とID、2位の点数とID…）。
- `score.item()` / `int(i)`：要素1個のPyTorchテンソルを、ふつうのPythonの小数・整数に変換します。
- `{...!r:8}` は `repr` 形式で幅8文字、`{:6.2f}` は小数第2位まで幅6文字で縦をそろえます。

</details>

#### 📝 確認問題（`ch03-q01` · Lv.2 · 辞書のアンパック **inputs の意味）

`model(**inputs)` の `**`（辞書アンパック）の動きを確認します。次のコードを実行すると何が表示されますか？

```python
def add_info(a, b):
    return a * 10 + b

data = {"a": 3, "b": 7}
print(add_info(**data))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：関数呼び出しの `**辞書`（辞書アンパック）は、キーを引数名・値を引数の値として展開します。
- **ヒント 2**：`add_info(**data)` は `add_info(a=3, b=7)` と同じ呼び出しになるので、`3 * 10 + 7` を計算します。
- **期待される出力**：`37`

**考え方**：関数呼び出しの `**辞書` は、辞書のキーを引数名、値を引数の値として展開して渡す記法です。

1. `data = {"a": 3, "b": 7}`。
1. `add_info(**data)` は `add_info(a=3, b=7)` に展開される。
1. `3 * 10 + 7 = 37` が出力される。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q01

</details>

#### 📝 確認問題（`ch03-q02` · Lv.2 · 出力テンソル [1, T, V] と [0, -1] の意味）

トークン数6個の文章を1件だけモデルに入力すると、`out.logits.shape` は `torch.Size([1, 6, 151936])` になります。そこから `out.logits[0, -1]` を取り出すと、何が得られますか？

- **a.** 入力した全6トークン位置それぞれの生の点数をすべて足し合わせた、文章全体の合計スコア
- **b.** 全151,936語の中で最も確率が高い1位候補の確率だけを表す、1個だけの小数（形 `[]`）
- **c.** 入力した文章の先頭（1文字目）に対応するトークンIDと、その文字のUTF-8バイト列のペア
- **d.** 0番目の文の末尾トークンの次に対する全151,936語の生の点数（形 `[151936]`）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：3次元テンソル `[バッチ数, トークン位置, 語彙数]` のどの軸を `[0, -1]` で固定しているか確認しましょう。
- **ヒント 2**：軸0を `0`（1件目の文）、軸1を `-1`（末尾の位置）に固定すると、残るのは軸2（全語彙の生の点数）です。
- **正解**：`d`

**考え方**：LLMは入力の各位置それぞれについて「その次のトークンの点数」を一度に出力します。文章の続きを予測したいので、一番最後の位置 `-1` を取り出します。

1. 軸0（長さ1）：文章の番号 → `0` で1件目の文章を選ぶ。
1. 軸1（長さ6）：トークンの位置（0〜5）→ `-1` で最後の位置（5番目＝「は」）を選ぶ。
1. 軸2（長さ151,936）：全語彙それぞれの生の点数 → 全部残るので形は `[151936]`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q02

</details>

*(この節の追加問題 `ch03-q03` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html で自動採点・復習できます)*

## Σ（シグマ）記号を普通の足し算から理解する

点数が `21.03` や `16.55` のままでは「何%の確率で選ばれるか」が分かりません。点数を確率に変える数式（softmax）を導く前に、式の中に登場する **$\sum$（シグマ）記号**をゼロから理解しましょう。

#### 📐 数学のポイント：Σ（シグマ）記号：9ステップで理由から理解する

**① 解決したい具体的な問題**：151,936個の数字を全部足す計算を、数式で短く正確に書きたい。

**② 普通の足し算では困る理由**：3個なら $z_1 + z_2 + z_3$ で済みますが、151,936個を書き並べるのは不可能です。「$\dots$」で省略すると、どこからどこまで何を足すのか曖昧になることがあります。

**③ 新しい発想**：「番号 $j$ を $1$ から $N$ まで1ずつ動かしながら、右側の式を全部足す」という専用の合図 **$\sum$**（ギリシャ文字の大文字で「**シグマ**」と読む。英語の Sum の S に相当）を使う。

**④ 各部分の読み方**：
$$\sum_{j=1}^{N} x_j$$
- $\sum$：「全部足せ」という命令
- 下の $j=1$：ループ変数 $j$ を `1` から始める
- 上の $N$：$j$ が `N` になるまで続ける（`N` を**含む**！）
- 右の $x_j$：各 $j$ について足す中身（例：$j$ 番目の数 $x_j$）

**⑤ 小さな数字で手計算**：$x = [10, 20, 30]$（$x_1=10, x_2=20, x_3=30$）のとき：
$$\sum_{j=1}^{3} x_j = x_1 + x_2 + x_3 = 10 + 20 + 30 = 60$$
式が $2j$ なら：
$$\sum_{j=1}^{3} (2j) = (2\times 1) + (2\times 2) + (2\times 3) = 2 + 4 + 6 = 12$$

**⑥ Pythonの `for` 文との対応**：$\sum$ はプログラムで書けばそのまま `for` 文です！

```python no-run
total = 0
for j in range(1, 4):   # 注意：Pythonの range(1, 4) は 1, 2, 3（4を含まない）
    total += 2 * j
print(total)            # 12
```

なお、範囲が文脈から明らかなときは上下を省略して $\sum_j e^{z_j}$（全部の $j$ について $e^{z_j}$ を足す）と書くこともよくあります。

確認問題で、手計算とPythonの両方から $\sum$ を定着させましょう！

#### 📝 確認問題（`ch03-q04` · Lv.1 · Σ記号の読み方と各部分の意味）

数式 $\sum_{j=1}^{4} (2j)$ の値を手計算で求めてください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：$\sum_{j=1}^{4}$ は「$j$ を 1 から 4 まで1ずつ増やしながら全部足す」という意味です。
- **ヒント 2**：$j = 1, 2, 3, 4$ をそれぞれ $2j$ に代入して4つの数を求め、合計しましょう。
- **正解**：`20`

**考え方**：$\sum_{j=1}^{4} (2j)$ は「$j$ を1から4まで1ずつ増やしながら $2j$ をすべて足す」という意味です。

1. $j=1$：$2 \times 1 = 2$
1. $j=2$：$2 \times 2 = 4$
1. $j=3$：$2 \times 3 = 6$
1. $j=4$：$2 \times 4 = 8$
1. 合計：$2 + 4 + 6 + 8 = 20$

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q04

</details>

#### 📝 確認問題（`ch03-q05` · Lv.2 · Σ記号とPythonのfor文の対応）

数式 $\sum_{j=1}^{4} (j + 1)$ をPythonの `for` 文で書いた次のコードの出力を予測してください。

```python
total = 0
for j in range(1, 5):
    total += j + 1
print(total)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Pythonの `range(1, 5)` がどの整数を順に出すか（終了値 `5` は含むか含まないか）を確認しましょう。
- **ヒント 2**：`j` は `1, 2, 3, 4` の4回回ります。それぞれ `j + 1` を計算して `total` に足し上げます。
- **期待される出力**：`14`

**考え方**：数学の $\sum_{j=1}^{N}$ は「$N$ を含む」のに対し、Pythonの `range(1, N + 1)` は終了値を含まない点に注意して対応づけます。

1. `j = 1` → `total = 0 + 2 = 2`
1. `j = 2` → `total = 2 + 3 = 5`
1. `j = 3` → `total = 5 + 4 = 9`
1. `j = 4` → `total = 9 + 5 = 14`

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q05

</details>

## なぜ素朴な割り算ではダメなのか？指数関数とsoftmaxの理由

いよいよ本題です。モデルが出した生の点数（ロジット）を、**確率**に変換します。ここでも「数式を暗記」ではなく、**「なぜその計算になるのか」**を1歩ずつたどりましょう。

#### 💡 理論のポイント：ステップ1〜3：問題 → 素朴な方法の失敗 → 指数関数 $e^z$ の発想

**① 解決したい問題**：たとえば「吾輩は○○である」の候補5つのロジットが `z = [3.0, 1.6, 1.2, 0.4, -1.0]` だとする。これを次の**確率の2条件**（＋順位の保存）を満たす数値に変えたい。
- 条件A：すべての確率 $p_i$ が **0以上**（マイナスの確率は存在しない）
- 条件B：すべての確率を足すと **ちょうど1**（$\sum_i p_i = 1$）
- 条件C：元の点数が高いほど、確率も高い（**順位が保たれる**）

**② 素朴な方法ではなぜダメなのか？**
- **案1：そのまま合計で割る** → `-1.0` の確率がマイナスになって条件Aに違反！ さらに `[2.0, -2.0]` のように合計が0だとゼロ除算で壊れる。
- **案2：全要素を2乗して合計で割る** → 全部0以上にはなるが、`-4.0` を2乗すると `+16.0` になり、`+3.0`（2乗すると `9.0`）より大きくなって**条件C（順位）が逆転**してしまう！
- **案3：最小値を引いて0以上にしてから割る** → 最小の候補が必ず確率ちょうど `0.0`（絶対に選ばれない）になり、全候補に同じ定数を足したり引いたりしたときの比率が不自然になる。

**③ 救世主の発想：指数関数 $e^z$（`math.exp(z)`）を使う！**
$e$（ネイピア数、約 `2.718`）を $z$ 乗した $e^z$ を各点数に掛けてから合計で割ると、すべての問題が一挙に解決します。

| $z$ | -2 | -1 | 0 | 1 | 2 | 3 |
|---|---|---|---|---|---|---|
| $e^z$ | 0.14 | 0.37 | 1.00 | 2.72 | 7.39 | 20.09 |

なぜ $e^z$ が理想的なのか？ 性質が4つあります。
1. **どんな $z$ でも必ずプラス**（$e^{-1} = 1/2.718 \approx 0.37 > 0$。マイナスにならない！）
2. **単調増加で順位を保つ**（$z$ が大きいほど $e^z$ も大きい）
3. **点数の「差」が確率の「倍率」になり、平行移動しても確率が変わらない**（指数法則 $e^{z_i + c} = e^c \cdot e^{z_i}$ より、全ロジットに $+100$ しても分子・分母の $e^{100}$ が約分されて確率が完全一致！）
4. **なぜ底が 2 や 10 ではなく $e$ なのか？**（先取りメモ）：$e^x$ は微分しても形がそのまま $(e^x)' = e^x$ という特別な性質を持ち、第7〜8章の学習（勾配計算）の式がもっとも単純できれいになるためです。

**④〜⑤ 小さな数字で手計算してみよう**
候補が3つで $z = [2.0, 1.0, 0.0]$ とします。
1. 各点数を $e^z$ にする：$e^2 \approx 7.39$、$e^1 \approx 2.72$、$e^0 = 1.00$
2. 合計（$\sum_j e^{z_j}$）を計算する：$7.39 + 2.72 + 1.00 = 11.11$
3. それぞれの $e^{z_i}$ を合計 `11.11` で割る：
   - $p_1 = 7.39 \div 11.11 \approx 0.665$（66.5%）
   - $p_2 = 2.72 \div 11.11 \approx 0.245$（24.5%）
   - $p_3 = 1.00 \div 11.11 \approx 0.090$（9.0%）
   - 合計：$0.665 + 0.245 + 0.090 = 1.000$！

**⑥ 数式への一般化（これが softmax！）**
以上の手順を、候補数 $V$ 個に対して一般化した式が **softmax** です。

$$p_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}$$

| 記号 | 読み方 | 意味 | Pythonでの対応 |
|---|---|---|---|
| $z_i$ | ゼット・アイ | $i$ 番目のトークンの生の点数（ロジット） | `logits_small[i]` |
| $e^{z_i}$ | イーの ゼット・アイ乗 | 点数を正の数に変換した値 | `math.exp(z)` |
| $\sum_{j=1}^{V} e^{z_j}$ | シグマ $j=1$ から $V$ まで $e^{z_j}$ | 全候補の $e^{z_j}$ の合計 | `sum(math.exp(z) for z in logits_small)` |
| $p_i$ | ピー・アイ | $i$ 番目のトークンが選ばれる確率 | `e / total` |

**⑦〜⑧ Pythonで実装して検証しよう**（まずは理解のための小さな5候補の純Pythonコード）：

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import math

words = ["猫", "犬", "人間", "名前", "机"]
logits_small = [3.0, 1.6, 1.2, 0.4, -1.0]

exps = [math.exp(z) for z in logits_small]
total = sum(exps)
probs_small = [e / total for e in exps]

for w, z, e, p in zip(words, logits_small, exps, probs_small):
    print(f"{w:3}  点数 {z:5.1f}  → exp {e:6.2f}  → 確率 {p:.3f}")
print("確率の合計：", sum(probs_small))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
猫    点数   3.0  → exp  20.09  → 確率 0.665
犬    点数   1.6  → exp   4.95  → 確率 0.164
人間   点数   1.2  → exp   3.32  → 確率 0.110
名前   点数   0.4  → exp   1.49  → 確率 0.049
机    点数  -1.0  → exp   0.37  → 確率 0.012
確率の合計： 1.0
```

</details>

注目してほしい点：「猫（3.0）」と「犬（1.6）」の点数差は `1.4` で、$e^{1.4} \approx 4.06$。実際の確率比も $0.665 \div 0.164 \approx 4.05$ 倍と一致しています！

では、本物のLLM（151,936語）のロジットを `torch.softmax` で確率に変えてみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
probs = torch.softmax(logits.float(), dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"{tok.decode([int(i)])!r:8} 確率 {p.item()*100:5.1f}%")
print("全トークンの確率の合計：", probs.sum().item())

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
'まだ'    確率  97.6%
'ない'    確率   1.1%
'、'      確率   0.3%
'「'      確率   0.3%
'無い'    確率   0.2%
全トークンの確率の合計： 1.0000001192092896
```

</details>

<details><summary><b>基礎の補足：なぜ `.float()` と `dim=-1` を書くのか？</b>（クリックで開く）</summary>

- **`.float()`**：GPUではモデルが `float16`（16ビット小数）で動いていますが、`float16` は最大約65,504までしか表せず、`exp` で簡単にオーバーフローしたり丸め誤差が大きくなったりします。`.float()` で32ビット小数（`float32`）にしてから `softmax` するのが安全な定石です。
- **`dim=-1`**：どの方向（軸）の合計を1にするかの指定です。`-1` は「一番最後の軸」を意味します。たとえば2行3列（形 `[2, 3]`）なら各行（長さ3の方向）ごとに合計1.0になり、1次元（形 `[151936]`）でもそのまま語彙全体の合計が1.0になります。
- なお合計が `1.0000001...` とごくわずかにずれるのは、コンピュータの小数の丸め誤差（浮動小数点数の仕様）です。

</details>

下のデモでも、ロジットを書き換えて確率の動きを体感してみてください！

*(対話デモはWeb教科書ページで確認できます：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html)*

ここでsoftmaxの理解を確認問題と自力実装でしっかり固めましょう！

#### 📝 確認問題（`ch03-q06` · Lv.2 · 素朴な割り算や2乗では確率に変換できない理由）

ロジット（生の点数）`[3.0, 1.0, -4.0]` を確率に変えるとき、「全部を2乗してから合計で割る」方法では**うまくいかない決定的な理由**はどれですか？

- **a.** 各候補を2乗してから全体の合計で割っても、確率の総和がどうしても `1.0` に一致しなくなるから
- **b.** マイナスの点数を2乗すると符号が負のまま大きくなり、確率の合計が必ずマイナスになってしまうから
- **c.** `-4.0` を2乗すると `+16.0` になり、最も低い点数だったはずの候補が一番高い確率になって順位が逆転するから
- **d.** PythonやPyTorchの仕様上、マイナスの小数に対して2乗（`** 2`）を計算することができないから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：点数を確率に変えるときに守るべき「大小順位の保存」が2乗で保たれるか確かめましょう。
- **ヒント 2**：元の点数で1位の `3.0` と最下位の `-4.0` をそれぞれ2乗した大きさを比べてみてください。
- **正解**：`c`

**考え方**：点数を確率に変える変換には「① 全部プラス（0以上）」「② 合計1」「③ 点数の大小順位を保つ（単調増加）」の3条件が必須です。2乗は③を破ります。

1. 元の点数：`3.0`（1位） > `1.0`（2位） > `-4.0`（最下位）。
1. 2乗すると：`9.0`, `1.0`, `16.0` → 最下位だった `-4.0` が1位になってしまう！
1. 一方、$e^z$ なら $e^{3.0} \approx 20.09 > e^{1.0} \approx 2.72 > e^{-4.0} \approx 0.018 > 0$ となり、全条件を満たす。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q06

</details>

#### 📝 確認問題（`ch03-q07` · Lv.1 · 指数関数 e^z の具体的な値（z=0, 1, -1））

$e^0$（$e$ の0乗）の値はいくつですか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：指数が1減るごとに元の値を $e$ で割る規則（$e^2 \to e^1 \to e^0$）を思い出しましょう。
- **ヒント 2**：$e^0 = e^1 \div e^1$ なので、0以外のどんな数でも0乗すると同じ1つの値になります。
- **正解**：`1`

**考え方**：指数法則 $e^1 \div e^1 = e^{1-1} = e^0 = 1$ より、$e^0 = 1$ です。これがロジット `0.0` のときの基準値 `1.0` になります。

1. $e^2 = e \times e \approx 7.39$
1. $e^1 = e \approx 2.72$
1. 1つ下がるごとに $e$ で割るので、$e^0 = e \div e = 1$
1. さらに下がると $e^{-1} = 1 \div e \approx 0.37$

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q07

</details>

*(この節の追加問題 `ch03-q08`, `ch03-q09`, `ch03-q10`, `ch03-q15` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html で自動採点・復習できます)*

## サイコロを振る：サンプリングと乱数シード

確率が分かったら、その確率に従ってトークンを1つ選びます（**サンプリング**）。0〜1の数直線を確率の幅で区切り、0以上1未満の一様乱数 `r` がどの区間に落ちたかで選びます。

<div class="math-display">0 ─── 猫（0.665）───┼─ 犬（0.164）─┼ 人間 ┼名前┼机┤ 1</div>

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import random

def sample(words, probs):
    r = random.random()          # 0以上1未満の乱数
    total = 0.0
    for w, p in zip(words, probs):
        total += p
        if r < total:
            return w
    return words[-1]

random.seed(0)
results = [sample(words, probs_small) for _ in range(1000)]
for w in words:
    print(w, results.count(w), "回")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
猫 664 回
犬 175 回
人間 104 回
名前 49 回
机 8 回
```

</details>

1000回振ると、猫が664回（約66.4%）と理論確率 `0.665` によく一致しました。

<details><summary><b>基礎の補足：`range`・`_`・`.count`・乱数シード</b>（クリックで開く）</summary>

- `for _ in range(1000)`：1000回繰り返す。`_` は「ループ番号自体は中で使わないよ」というPythonの慣例名です。
- `results.count(w)`：リストの中に `w` が何個あるかを数えます。
- `random.seed(0)`：擬似乱数の初期値（**シード**）を固定します。シードを同じにすれば同じ乱数列が再現されるため、実験比較では必ずシードを固定・記録します。
- PyTorchでは `torch.multinomial(probs, num_samples=1)` が同じくじ引きを高速に行ってくれます。

</details>

#### 📝 確認問題（`ch03-q11` · Lv.3 · 累積和によるサンプリングのトレース）

確率が `猫: 0.60`, `犬: 0.25`, `机: 0.15` のとき、0〜1の乱数 `r` が **`0.72`** だったとします。次のサンプリング関数が返す単語を答えてください。

```python
words = ["猫", "犬", "机"]
probs = [0.60, 0.25, 0.15]
r = 0.72
total = 0.0
for w, p in zip(words, probs):
    total += p
    if r < total:
        print(w)
        break
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：累積和 `total` を1周目から順に足し上げ、最初に `r < total` が成り立つ単語を探します。
- **ヒント 2**：1周目の `total` は `0.60`、2周目は `0.60 + 0.25 = 0.85` です。`r = 0.72` が最初に下回るのはどの周でしょう？
- **期待される出力**：`犬`

**考え方**：0〜1の線分を `[0, 0.60)`＝猫、`[0.60, 0.85)`＝犬、`[0.85, 1.00]`＝机 に区切ったとき、`0.72` は犬の区間に落ちます。

1. 1周目（猫）：`total = 0.60`。`0.72 < 0.60` は `False`。
1. 2周目（犬）：`total = 0.60 + 0.25 = 0.85`。`0.72 < 0.85` は `True` → `犬` を出力して終了！

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q11

</details>

#### 📝 確認問題（`ch03-q12` · Lv.1 · 乱数シードを固定する理由）

実験や研究で `random.seed(0)` や `torch.manual_seed(42)` のように**シード（種）を固定して記録する**一番の目的は何ですか？

- **a.** サンプリングを無効にして、毎回かならず確率1位のトークンだけを選ぶ貪欲法（greedy）へ切り替えるため
- **b.** GPUの計算コア数を制限して消費電力やVRAM使用量を節約し、Colabの制限時間を延ばすため
- **c.** シード値に `42` などの大きな数を渡すことで、モデルのパラメータ数や推論精度を自動的に向上させるため
- **d.** 乱数を使う実験でも同じ結果を再現できるようにし、設定変更の効果を公平かつ正確に比較できるようにするため

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：温度を変えた比較実験で、乱数が毎回バラバラだったら結果の差が何のせいか判断できるか考えましょう。
- **ヒント 2**：シードを固定すると「同じ乱数列が再現される」ため、条件変更だけの純粋な影響を比較できます。
- **正解**：`d`

**考え方**：コンピュータの乱数は計算式で作る擬似乱数なので、同じシードを与えれば同じ乱数列が再現されます。

1. シードなし：実行のたびに違う乱数列 → 比較実験でブレる。
1. シード固定：同じ条件なら同じ結果が再現できる。
1. 論文や研究では複数シードで試して平均とばらつきを確認するのも重要（第20章）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q12

</details>

*(この節の追加問題 `ch03-q18` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html で自動採点・復習できます)*

## 温度：LLMの「はっちゃけ度」

LLMの創造性や堅実さを調整するつまみが **温度**（temperature、記号 $T$）です。仕組みは驚くほど単純で、**softmaxの前にすべてのロジットを $T$ で割る**だけです。

$$p_i = \frac{e^{z_i / T}}{\sum_{j=1}^{V} e^{z_j / T}}$$

#### 💡 理論のポイント：なぜ $T$ で割ると性格が変わるのか？（小さな数字で検証）

2つの候補のロジットが $3.0$ と $1.0$（差は $2.0$）だとします。

| 温度 $T$ | 割った後の点数 | 点数の差 | 確率の倍率 $e^{\text{差}}$ | 振る舞い |
|---|---|---|---|---|
| $T = 0.5$（低温） | `6.0` と `2.0` | **`4.0`（2倍に拡大）** | $e^4 \approx 54.6$ 倍 | 1位がほぼ独占。堅実だが単調になりやすい |
| $T = 1.0$（標準） | `3.0` と `1.0` | `2.0`（そのまま） | $e^2 \approx 7.39$ 倍 | モデルが学んだそのままの確率分布 |
| $T = 2.0$（高温） | `1.5` と `0.5` | **`1.0`（半分に縮小）** | $e^1 \approx 2.72$ 倍 | 確率が平らになり、珍しい語も選ばれやすい |
| $T \to 0$ の極限 | — | $\infty$ | 常に1位のみ（**greedy**） | 第1章の `do_sample=False`（実装上は `argmax` を使う） |

※ 温度に `0` をそのまま入れるとゼロ除算になるため、greedyにしたいときは `argmax` を使います。

Pythonでも5候補の確率変化を確認しましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def softmax_t(logits, T):
    exps = [math.exp(z / T) for z in logits]
    total = sum(exps)
    return [e / total for e in exps]

for T in [0.3, 1.0, 2.0]:
    p = softmax_t(logits_small, T)
    print(f"T={T}: " + "  ".join(f"{w}{x:.2f}" for w, x in zip(words, p)))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
T=0.3: 猫0.99  犬0.01  人間0.00  名前0.00  机0.00
T=1.0: 猫0.66  犬0.16  人間0.11  名前0.05  机0.01
T=2.0: 猫0.43  犬0.21  人間0.18  名前0.12  机0.06
```

</details>

#### 📝 確認問題（`ch03-q13` · Lv.2 · 温度 T でロジットを割ると確率分布がどう変わるか）

2つの候補のロジットが `3.0` と `1.0`（差は `2.0`）です。温度を **`T = 0.5`** にしたとき、温度で割ったあとの2つの点数の**差**はいくつになりますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：温度 `T` を適用するとき、各ロジットを `T` で割ってからsoftmaxに入れます。`0.5` で割ると値は大きくなるでしょうか、小さくなるでしょうか？
- **ヒント 2**：それぞれ `0.5` で割ると `3.0 / 0.5` と `1.0 / 0.5` になります。割ったあとの2つの値の差を求めましょう。
- **正解**：`4.0`

**考え方**：1より小さい数 $T = 0.5$ で割ることは「2倍する」のと同じなので、点数の差が2倍に拡大し、$e^{\text{差}}$ の倍率が急激に広がって1位に確率が集中します。

1. 割る前：$3.0 - 1.0 = 2.0$（倍率 $e^2 \approx 7.39$ 倍）。
1. $T = 0.5$ で割る：$3.0 / 0.5 = 6.0$、$1.0 / 0.5 = 2.0$。
1. 割った後の差：$6.0 - 2.0 = 4.0$（倍率 $e^4 \approx 54.6$ 倍！）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q13

</details>

#### 📝 確認問題（`ch03-q14` · Lv.2 · 温度 T=0 でエラーになる理由と greedy の関係）

自作の `softmax_t(logits, T)` にそのまま `T = 0` を渡すとエラーになる理由と、温度→0の極限と同じ結果を得る正しい方法の組み合わせはどれですか？

- **a.** 指数関数 $e^0$ は数学的に定義されていないためエラーになり、回避するには全ロジットに `1` を足す必要がある
- **b.** `T = 0` でもゼロ除算エラーは一切起きず、安全装置として常に空文字だけを返して正常終了する仕様になっている
- **c.** `T = 0` では全候補の確率が均等になりすぎるため、代わりに `T = 100` を指定すればgreedyと同じ結果になる
- **d.** `z / T` でゼロ除算エラーになる。同じgreedy結果を得るには `argmax` で最大点数のIDを選ぶ

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：温度つきsoftmaxの式 $p_i = e^{z_i / T} / \sum_j e^{z_j / T}$ で $T = 0$ を代入したとき、最初に困る演算を探しましょう。
- **ヒント 2**：`z / 0` はゼロ除算になるため、プログラムでは $T=0$ のときだけ最大スコアのIDを直接選ぶ関数へ分岐します。
- **正解**：`d`

**考え方**：数式上は $T \to 0^+$ の極限が「最大値1位を確率1で選ぶ（argmax / greedy）」に対応しますが、プログラムでは `T == 0` を直接割れないため分岐処理で `argmax` を呼びます。

1. `z / 0` はゼロ除算になる。
1. 数学的には $T$ を0に近づけるほど1位の確率が1.0に近づく。
1. コードでは `if temperature == 0: next_id = torch.argmax(logits)` と書く（第1章の `do_sample=False` と同じ考え方）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q14

</details>

*(この節の追加問題 `ch03-q19` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html で自動採点・復習できます)*

## 自分で文章を生成する

いよいよ「点数 → 温度つきsoftmax → サンプリング → 末尾に付け足す」をつないで、自分の手で文章生成関数 `generate` を書きます！

<details><summary><b>基礎の補足：実行前に確認：自作 `generate` の1行ごとのテンソル（数の表、第6章）shape 追跡</b>（クリックで開く）</summary>

「吾輩は猫である。名前は」（長さ6トークン）を入れた1周目の流れです。

| 処理 | 変数 | 形（`shape`） | 何をしているか |
|---|---|---|---|
| `tok(text, return_tensors="pt").input_ids.to(...)` | `ids` | `[1, 6]` | 文章をトークンIDの2次元テンソル（1文章×6トークン）へ |
| `model(ids).logits[0, -1]` | `logits` | `[151936]` | 末尾位置の次トークンの生の点数を取り出す |
| `torch.softmax(logits.float() / temperature, dim=-1)` | `probs` | `[151936]` | 温度で割って確率分布（合計1）に変換 |
| `torch.multinomial(probs, num_samples=1)` | `next_id` | `[1]` | 確率に従ってトークンIDを1個サンプリング |
| `next_id.view(1, 1)` | — | `[1, 1]` | `ids`（2次元）と連結できるよう形を `[1, 1]` に整える |
| `torch.cat([ids, next_id.view(1, 1)], dim=1)` | `ids` | `[1, 7]` | 軸1（横方向）に連結して文章を1トークン伸ばす |

（`.view` やテンソル操作の詳細は第6章・第9章で本格的に練習します。ここでは「毎ステップ `[1, 6] → [1, 7] → [1, 8]` と1つずつ伸びる」ことをつかめれば完璧です！）

</details>

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def generate(text, n_tokens=30, temperature=1.0):
    ids = tok(text, return_tensors="pt").input_ids.to(model.device)
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]               # ① 次のトークンの点数
        probs = torch.softmax(logits.float() / temperature, dim=-1)  # ② 温度つきで確率に
        next_id = torch.multinomial(probs, num_samples=1)   # ③ サイコロを振る
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)   # ④ 最後に付け足す
    return tok.decode(ids[0])

torch.manual_seed(42)
print(generate("吾輩は猫である。名前は", n_tokens=40, temperature=0.7))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
吾輩は猫である。名前はまだ無い。どこで生れたかとんと見当がつかぬ。何でも薄暗いじめじめした所でニャーニャー泣いていた事だけは記憶している。
```

</details>

**ボンゴ先生**：見たかい？ ChatGPTを含む現代のすべてのLLMの文章生成の骨組みは、この10行なんだ。1つ予測して、自分の出力を次の入力の末尾に付け足して、また予測する──これを **自己回帰（autoregressive）生成** と呼ぶよ！

> **💡 理論のポイント：自己回帰生成の弱点と、先の章へのつながり**
>
> この素朴なループでは、トークンを1個増やすたびにそれまでの文章全体を毎回モデルに通し直すため、文章が長くなるほど1トークンあたりの計算が遅くなります。実際のLLMでは「過去の計算結果を覚えておいて再利用する」工夫（**KVキャッシュ**）で高速化します（第16章で自分で実装します）。

#### 🔬 やってみよう：温度を変えて書き比べよう

同じ書き出し・同じシードで温度だけ変えて実行してみましょう。先に「どの温度が一番暴走しそうか」予想してから試します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
for T in [0.1, 0.7, 1.5]:
    torch.manual_seed(0)
    print(f"--- 温度 {T} ---")
    print(generate("ある日、AIが", n_tokens=40, temperature=T))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
--- 温度 0.1 ---
ある日、AIが人間の仕事を奪うのではないかという不安が広がっていました。しかし、AIは人間の仕事を奪うのではなく、
--- 温度 0.7 ---
ある日、AIが突然「私は人間になりたい」と言い出した。研究者たちは驚き、その理由を尋ねた。AIは静かに答えた。
--- 温度 1.5 ---
ある日、AIが鍋焼きうどんに論文を提出、なお昨年度の宇宙会計は猫部に移管されました。ふと見上げると
```

</details>

最後に、`transformers` 標準の `model.generate(...)` と見比べましょう（標準版にはKVキャッシュや、第16章で学ぶ `top-p` などの安全装置も組み込まれています）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
inputs = tok("吾輩は猫である。名前は", return_tensors="pt").to(model.device)
torch.manual_seed(42)
out = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=0.7)
print(tok.decode(out[0], skip_special_tokens=True))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
吾輩は猫である。名前はまだ無い。どこで生れたかとんと見当がつかない。ただ、暗い場所で鳴いていたことだけは覚えている。
```

</details>

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`RuntimeError: Expected all tensors to be on the same device`**：入力とモデルの置き場所（CPU/GPU）がずれています。`.to(model.device)` を付け忘れていないか確認しましょう。
> - **`probability tensor contains either inf, nan or element < 0`**：温度に `0` を入れてゼロ除算したときに起きます。greedyにしたいときは `torch.argmax(logits)` を使います。
> - **同じ言葉のくり返し**：低い温度や小さなモデルではよく起きます（第16章で繰り返しを抑える工夫を学びます）。

第3章と第1部全体のまとめ問題に挑戦しましょう！

#### 📝 確認問題（`ch03-q16` · Lv.3 · 自作 generate ループのテンソル shape 変化の追跡）

自作 `generate` 関数で、最初の入力トークン列 `ids` の形が `[1, 6]`（1文章 × 6トークン）でした。ループを **1回** 回して `ids = torch.cat([ids, next_id.view(1, 1)], dim=1)` を実行した直後、`ids.shape` は何になりますか？

- **a.** `[1, 7]`（末尾に新トークンIDが1個連結され長さ7になる）
- **b.** `[2, 6]`（トークン長は6のままで、バッチ内の文章の個数が1から2に増える）
- **c.** `[1, 6]`（古い先頭トークンが自動で消えるため、テンソルの形は変わらない）
- **d.** `[1, 151936]`（語彙全体のロジット配列がそのまま連結されて横幅が語彙数になる）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`torch.cat(..., dim=1)` が2つの軸（軸0＝文章数、軸1＝トークン長）のどちらを伸ばすか確認しましょう。
- **ヒント 2**：`[1, 6]` と `[1, 1]` を `dim=1`（2つ目の軸）に沿ってつなぐと、軸1の長さが `6 + 1` になります。
- **正解**：`a`

**考え方**：自己回帰生成では、1ステップごとに選んだ1トークンを形 `[1, 1]` に整えて `dim=1` で連結し、`[1, 6] → [1, 7] → [1, 8] ...` と文章を伸ばしていきます。

1. `ids`：`[1, 6]`（1文章、6トークン）。
1. `next_id`：`[1]` → `.view(1, 1)` で2次元の `[1, 1]` に形を合わせる（`cat` は次元数が同じでないと連結できない）。
1. `torch.cat([ids, next_id.view(1, 1)], dim=1)` → `[1, 7]`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q16

</details>

#### 📝 確認問題（`ch03-q17` · Lv.2 · 自己回帰生成の意味と長文で遅くなる理由）

自作した `generate` ループの方式は「自己回帰（autoregressive）生成」と呼ばれます。なぜそう呼ばれるのか、また文章が長くなるほど1トークンあたりの計算が重くなるのはなぜか説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：生成されたトークンが次のステップでどこに入るか、そして毎回の `model(ids)` に何を渡しているかを追いましょう。
- **ヒント 2**：「自分の出力を次の入力の末尾に付け足して繰り返す」「毎回先頭からの文章全体をモデルに通し直しているため長文ほど重い」の2点をまとめます。
- **解答の目安**：モデル自身が予測して出力したトークンを、次のステップの入力の末尾に付け足して再びモデルへ入力する（自分の出力が次の自分の入力になる）ため「自己回帰」と呼びます。素朴なループでは1トークン増えるたびに先頭からの文章全体を毎回モデルに通し直すため、文章が長くなるほど計算量が増えて遅くなります。

**考え方**：この「毎回先頭から再計算してしまう無駄」をなくすための高速化技術が、第16章で自作する **KVキャッシュ** です。

1. ステップ1：長さ6の文をモデルに通す → 7個目を得る。
1. ステップ2：長さ7の文（自分の出力込み）をモデルに通す → 8個目を得る。
1. 毎回先頭から計算し直すため、長くなるほど重くなる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html#ex-ch03-q17

</details>

*(この節の追加問題 `ch03-q20` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html で自動採点・復習できます)*

**✅ できたらチェック**

- [ ] `model(**inputs)`、`with torch.no_grad():`、`out.logits[0, -1]` の意味と形を説明できる
- [ ] $\sum$ 記号を手計算し、`for` 文へ書き換えられる
- [ ] なぜsoftmaxで素朴な割り算や2乗ではなく $e^z$ を使うのかを説明でき、オーバーフロー安全な `safe_softmax` を自力実装した
- [ ] 累積区間サンプリングと温度 $T$ の働きを説明でき、自作 `generate` で文章を生成できた

> **📝 まとめ**
>
> - モデルは入力 `[1, T]` に対し、各位置の次トークン点数（ロジット）`[1, T, V]` を返す。末尾 `[0, -1]` が続きの予測
> - $\sum$（シグマ）は「全部足す」の速記法で、Pythonの `for` 文に対応する
> - **softmax**：$p_i = e^{z_i/T} / \sum_j e^{z_j/T}$。$e^z$ により全要素が正・順位保存・平行移動不変になり、最大値を引けばオーバーフローも防げる
> - 確率に従ってくじを引き（サンプリング）、末尾に連結して繰り返すのが **自己回帰生成**

**ボンゴ先生**：第1部の完走おめでとう！ ここまでで「LLMを使う側・中身をのぞく側」の基礎はバッチリだ。しかも数式の理由も、コードの型と形も、自力実装までやり切ったね。<br>次の第2部（第4章〜）からは、いよいよ**モデル本体をゼロから作る側**に入る。まずは夏目漱石の『こころ』を材料に、「数を数えるだけ」の言語モデル（条件付き確率）からスタートしよう！

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html
- **次の章**：[第4章 数えるだけで漱石っぽくなる：bigramモデル](https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html)